# Homework 2 — Convolutional Neural Networks

This notebook mirrors the four sections of the PDF:

1. **Convolution Fundamentals** — parameter counting, manual 2D convolution, output-size formula, edge detectors, padding, receptive field.
2. **Pooling** — max vs average pooling, effect on receptive field.
3. **Architecture Analysis** — LeNet-5 parameter accounting, why 3×3 filters dominate.
4. **Backpropagation Through Conv Layers** — gradients with respect to kernel and input, max- and avg-pool gradients (sparse vs dense).

A final **Bonus** section trains a small CNN on MNIST end-to-end.

All numerical answers should match the PDF exactly. Run the notebook top-to-bottom; outputs are reproducible thanks to fixed random seeds.

In [ ]:
# ============================================================================
# Imports and reproducibility
# ============================================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Fixed seeds so all outputs (random pixels, weight init, etc.) are reproducible.
np.random.seed(42)
torch.manual_seed(42)

# Slightly nicer numpy / torch printing.
np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3)

print(f"PyTorch: {torch.__version__}, NumPy: {np.__version__}")


---
## Section 1 — Convolution Fundamentals


### Problem 1 — FC vs Conv parameter counting [5 pts]

Compare a fully-connected layer mapping a flattened $32 \times 32 \times 3$ image to 128 hidden units versus a Conv2d layer with 16 filters of size $5 \times 5$.

**Expected answers** (from the PDF):
- FC: $128 \cdot 3072 + 128 = 393{,}344$
- Conv: $16 \cdot (5 \cdot 5 \cdot 3 + 1) = 1{,}216$
- Ratio: $\approx 323\times$


In [ ]:
# TODO (a): write a helper that counts learnable parameters in a module.
def count_params(module):
    # Total number of learnable parameters in a module.
    raise NotImplementedError("implement count_params")

# TODO (b): build the fully-connected layer (nn.Linear) and print its param count.
fc_layer = ...     # replace with nn.Linear(...)

# TODO (c): build the Conv2d layer (nn.Conv2d) and print its param count.
conv_layer = ...   # replace with nn.Conv2d(...)

# TODO (d): print the ratio and verify against the expected values.


**Why this works:** Locality + weight sharing. The FC layer connects every output to every input pixel; the Conv layer reuses the same $5 \times 5$ filter at every spatial position. Parameter count for Conv depends only on filter size and channel counts — *not* on the image's spatial dimensions.

### Problem 2 — 2D convolution by hand [8 pts]

Implement 2D cross-correlation from scratch with nested loops, then verify against `torch.nn.functional.conv2d`.

**Setup** (matches the PDF's example):
$$
\mathbf{X} = \begin{pmatrix} 1 & 2 & 3 & 0 & 1 \\ 0 & 1 & 2 & 3 & 1 \\ 2 & 1 & 0 & 1 & 2 \\ 1 & 0 & 2 & 3 & 0 \\ 0 & 1 & 1 & 2 & 1 \end{pmatrix}, \quad
\mathbf{K} = \begin{pmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{pmatrix}
$$


In [ ]:
# TODO (a): implement a 2D cross-correlation function.
#   Inputs:  X (2D array), K (2D kernel), stride (int), padding (int)
#   Output:  Y of shape ((H + 2P - F)/S + 1, (W + 2P - F)/S + 1)
#   Hint:    use np.pad for padding, then nested loops over output positions.
def conv2d_naive(X, K, stride=1, padding=0):
    raise NotImplementedError("implement conv2d_naive")


# Test inputs (DO NOT MODIFY)
X = np.array([
    [1, 2, 3, 0, 1],
    [0, 1, 2, 3, 1],
    [2, 1, 0, 1, 2],
    [1, 0, 2, 3, 0],
    [0, 1, 1, 2, 1],
], dtype=float)

K = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
], dtype=float)

# TODO (b): apply your conv2d_naive with stride=1, padding=0; print the result.


In [ ]:
# (b) Verify against PyTorch's conv2d (this should pass once you fill in conv2d_naive)
X_t = torch.from_numpy(X).float().reshape(1, 1, 5, 5)
K_t = torch.from_numpy(K).float().reshape(1, 1, 3, 3)
Y_torch = F.conv2d(X_t, K_t, stride=1, padding=0).squeeze().numpy()

print("PyTorch output:")
print(Y_torch)
try:
    print(f"\nManual matches PyTorch: {np.allclose(Y, Y_torch)}")
except NameError:
    print("\nDefine Y in the previous cell first.")


In [ ]:
# TODO (c): apply conv2d_naive with stride=2 and padding=1. Print the result and shape.


### Problem 3 — Output size formula [4 pts]

Verify the formula
$$ W_{\text{out}} = \left\lfloor \frac{W_{\text{in}} + 2P - F}{S} \right\rfloor + 1 $$
on several test cases, then compute the parameter count for a multi-channel layer.


In [ ]:
# TODO: implement output_size(W, F, P, S) using the formula above.
def output_size(W, F, P, S):
    raise NotImplementedError("implement output_size")


# Test cases (DO NOT MODIFY -- these come from the PDF)
test_cases = [
    # (W,   F, P, S, expected)
    (28,   3, 0, 1, 26),
    (28,   3, 1, 1, 28),
    (28,   5, 2, 1, 28),
    (224,  3, 1, 2, 112),
    (224,  7, 3, 2, 112),
]
print(f"{'W':>4} {'F':>3} {'P':>3} {'S':>3}   {'computed':>9} {'expected':>9}  match")
print("-" * 50)
for W, F_, P, S, expected in test_cases:
    out = output_size(W, F_, P, S)
    print(f"{W:>4} {F_:>3} {P:>3} {S:>3}   {out:>9} {expected:>9}  {out == expected}")


In [ ]:
# TODO: write a helper conv_params(F, C_in, C_out) that returns the parameter
# count for a Conv2d layer with FxF kernel, including biases.
def conv_params(F, C_in, C_out):
    raise NotImplementedError("implement conv_params")

# TODO: apply to ResNet-style first layer (7x7 kernel, 3 -> 64 channels) and verify.


### Problem 4 — Edge detector demo [4 pts]

Hand-design vertical and horizontal edge detector kernels (Sobel-style) and apply them to a synthetic image with both kinds of edges.


In [ ]:
# Synthetic image: bright on the right, slightly brighter on the bottom (DO NOT MODIFY)
img = np.zeros((20, 20))
img[:, 10:] = 1.0
img[10:, :] = img[10:, :] + 0.5

# TODO: design a 3x3 Sobel-style vertical edge detector (responds to intensity
# changes left-to-right). Standard form has +1/+2/+1 in one column, -1/-2/-1
# in the other, and zeros in the middle.
K_v = np.zeros((3, 3))   # replace with your kernel

# TODO: derive the horizontal edge detector from K_v (hint: transpose).
K_h = np.zeros((3, 3))   # replace with your kernel

# Apply both detectors using your conv2d_naive from Problem 2.
out_v = conv2d_naive(img, K_v)
out_h = conv2d_naive(img, K_h)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1.5); axes[0].set_title("Input image")
axes[1].imshow(out_v, cmap="RdBu_r");                axes[1].set_title("Vertical edge detector")
axes[2].imshow(out_h, cmap="RdBu_r");                axes[2].set_title("Horizontal edge detector")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


### Problem 5 — Padding modes [4 pts]

Compare *valid*, *same*, and *full* convolutions on the same input. For stride 1 and odd kernel size $F$, *same* padding requires $P = (F-1)/2$.


In [ ]:
# Same input, three different padding choices (DO NOT MODIFY the input)
np.random.seed(0)
X = np.random.rand(8, 8)
F_size = 3
K_uniform = np.ones((F_size, F_size)) / F_size**2  # box blur

# TODO: apply conv2d_naive with three padding choices and print the output shapes.
#   Valid: P = 0
#   Same:  P = (F - 1) / 2 (preserves input size when stride is 1)
#   Full:  P = F - 1


### Problem 6 — Receptive field walkthrough [4 pts]

The receptive field of a neuron at layer $\ell$ is governed by the recurrence:
$$
j_\ell = j_{\ell-1} \cdot S_\ell, \qquad
\text{RF}_\ell = \text{RF}_{\ell-1} + (F_\ell - 1) \cdot j_{\ell-1}
$$
with $j_0 = 1$ and $\text{RF}_0 = 1$. Pooling layers double $j$, which then *multiplies* the receptive field gain of all subsequent conv layers.


In [ ]:
# TODO: implement compute_rf(layers) using the recurrences above.
#   `layers` is a list of (F, S) tuples.
#   Return a list of dicts with keys: layer, op, j, RF.
def compute_rf(layers):
    raise NotImplementedError("implement compute_rf")


# 5-layer network from the PDF:
#  Conv 5x5  -> Pool 2x2 s=2 -> Conv 5x5 -> Pool 2x2 s=2 -> Conv 3x3
network = [(5, 1), (2, 2), (5, 1), (2, 2), (3, 1)]
history = compute_rf(network)

# Print a table of j and RF at each layer. Final RF should be 24.
print(f"{'Layer':>5}  {'Operation':<14}  {'j':>3}  {'RF':>3}")
print("-" * 35)
for h in history:
    print(f"{h['layer']:>5}  {h['op']:<14}  {h['j']:>3}  {h['RF']:>3}")
print(f"\nFinal receptive field: {history[-1]['RF']} x {history[-1]['RF']} pixels")


---
## Section 2 — Pooling


### Problem 7 — Max vs average pooling [4 pts]

Apply $2 \times 2$ max and average pooling (stride 2) to a $4 \times 4$ matrix, then verify with PyTorch.


In [ ]:
X = np.array([
    [2, 5, 1, 3],
    [4, 1, 8, 0],
    [7, 2, 1, 4],
    [3, 0, 6, 9],
], dtype=float)

# TODO: implement 2x2 max pooling with stride 2.
def maxpool2x2(X):
    raise NotImplementedError("implement maxpool2x2")

# TODO: implement 2x2 average pooling with stride 2.
def avgpool2x2(X):
    raise NotImplementedError("implement avgpool2x2")

print("Max pool (manual):")
print(maxpool2x2(X))
print("\nAvg pool (manual):")
print(avgpool2x2(X))

# Verify with PyTorch
X_t = torch.from_numpy(X).float().reshape(1, 1, 4, 4)
print("\nMax pool (PyTorch):")
print(F.max_pool2d(X_t, 2).squeeze().numpy())
print("\nAvg pool (PyTorch):")
print(F.avg_pool2d(X_t, 2).squeeze().numpy())


**When to use which.** Max pool is the standard choice for detection-like tasks (it preserves the strongest local activation). Average pool is preferred for the *final* aggregation in modern architectures (e.g., global average pooling in ResNet) because it retains more information.

### Problem 8 — Pooling effect on receptive field [4 pts]

A network of two conv layers, then a max-pool, then two more conv layers. Reuse the recurrence from Problem 6.


In [ ]:
# Conv 3x3 -> Conv 3x3 -> MaxPool 2x2 (s=2) -> Conv 3x3 -> Conv 3x3
network = [(3, 1), (3, 1), (2, 2), (3, 1), (3, 1)]
history = compute_rf(network)

print(f"{'Layer':>5}  {'Operation':<14}  {'j':>3}  {'RF':>3}")
print("-" * 35)
for h in history:
    print(f"{h['layer']:>5}  {h['op']:<14}  {h['j']:>3}  {h['RF']:>3}")
print(f"\nFinal receptive field: {history[-1]['RF']} x {history[-1]['RF']}")
print("\nObservation: the maxpool only added 1 to RF directly, but DOUBLED")
print("the jump factor j. Subsequent conv layers now contribute 4 to RF")
print("each (not 2), because each kernel step covers 2 input pixels.")


---
## Section 3 — Architecture Analysis


### Problem 9 — LeNet-5 parameter count [7 pts]

Build the original LeNet-5 in PyTorch and verify the **61,706** parameter total. Expected breakdown:
- Conv $5\times5$, 1 → 6:  $5\cdot 5\cdot 1\cdot 6 + 6 = 156$
- Conv $5\times5$, 6 → 16: $5\cdot 5\cdot 6\cdot 16 + 16 = 2{,}416$
- FC $400 \to 120$: $48{,}120$
- FC $120 \to 84$:  $10{,}164$
- FC $84 \to 10$:    $850$

Total: 61,706.


In [ ]:
# TODO: build the LeNet-5 architecture as a torch.nn.Module.
# Input shape is expected to be (batch, 1, 32, 32).
# Layers (in order):
#   Conv2d(1 -> 6, kernel=5)        -> tanh -> AvgPool 2x2 stride 2
#   Conv2d(6 -> 16, kernel=5)       -> tanh -> AvgPool 2x2 stride 2
#   Flatten
#   Linear(16*5*5 -> 120)           -> tanh
#   Linear(120 -> 84)               -> tanh
#   Linear(84 -> 10)
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define layers as attributes
        raise NotImplementedError("implement LeNet5.__init__")

    def forward(self, x):
        # TODO: forward pass
        raise NotImplementedError("implement LeNet5.forward")


lenet = LeNet5()
print("LeNet-5 parameter breakdown:")
print("-" * 50)
total = 0
for name, layer in lenet.named_children():
    n = count_params(layer)
    print(f"  {name:>6}: {n:>10,} parameters")
    total += n
print("-" * 50)
print(f"  {'TOTAL':>6}: {total:>10,} parameters")
print(f"\nMatches PDF expected value (61,706): {total == 61706}")


In [ ]:
# Compare with an MLP that achieves similar MNIST accuracy
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(32 * 32, 300), nn.Tanh(),
    nn.Linear(300, 300),     nn.Tanh(),
    nn.Linear(300, 10),
)
mlp_params = count_params(mlp)
print(f"Equivalent MLP (1024 -> 300 -> 300 -> 10): {mlp_params:,} parameters")
print(f"MLP / LeNet-5 ratio: {mlp_params / total:.2f}x more parameters in the MLP")


### Problem 10 — Why $3 \times 3$ filters dominate [7 pts]

Compare two architectures with the **same** receptive field of $5 \times 5$:
- A: one $5 \times 5$ conv, $C \to C$
- B: two stacked $3 \times 3$ convs, each $C \to C$

B has fewer parameters (about 72% of A) and twice as many nonlinearities (two ReLUs instead of one), for the same RF.


In [ ]:
C = 64  # channels in and out

# TODO (a): build Architecture A: a single 5x5 convolution, C -> C, padding=2.
arch_a = ...  # nn.Conv2d(...)

# TODO (b): build Architecture B: two stacked 3x3 convolutions with a ReLU
# between them, each C -> C, padding=1.
arch_b = ...  # nn.Sequential(...)

# TODO (c): print the parameter counts and the ratio.


In [ ]:
# Generalize: stack n 3x3 convs has RF = 2n + 1
# Compare with one (2n+1) x (2n+1) conv at the same RF.
print(f"{'n':>2}  {'RF':>3}  {'stacked 3x3 params':>20}  {'single conv params':>20}  {'savings':>10}")
print("-" * 65)
for n in [1, 2, 3, 4, 5]:
    rf = 2 * n + 1
    p_stacked = n * (3 * 3 * C * C)         # bias ignored for simplicity
    p_single  = rf * rf * C * C
    savings = 100 * (1 - p_stacked / p_single)
    print(f"{n:>2}  {rf:>3}  {p_stacked:>20,}  {p_single:>20,}  {savings:>9.0f}%")


---
## Section 4 — Backpropagation Through Conv Layers


### Problem 11 — Gradient with respect to the kernel [10 pts]

For a 1D convolution $y_i = \sum_{u} x_{i+u} K_u$, the gradient with respect to the kernel is itself a convolution:
$$
\frac{\partial \mathcal{L}}{\partial K_u} = \sum_i \delta_i \, x_{i+u}
\qquad \text{where} \qquad \delta_i = \frac{\partial \mathcal{L}}{\partial y_i}
$$

Compute this manually, then verify against PyTorch's autograd.


In [ ]:
# Tiny example: x of length 5, kernel of length 3 (DO NOT MODIFY)
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
K = np.array([0.5, -0.5, 1.0])
n, k = len(x), len(K)

# TODO (a): forward pass -- compute y_i = sum_u x_{i+u} * K_u for i = 0..n-k.
y = np.zeros(n - k + 1)
# fill in y...

# Suppose the upstream gradient is delta = [1, 1, 1] (this is what dL/dy looks like
# if the loss is sum(y)).
delta = np.ones(n - k + 1)

# TODO (b): manually compute dL/dK using the formula
#   dL/dK_u = sum_i delta_i * x_{i+u}
dK_manual = np.zeros(k)
# fill in dK_manual...

print(f"Manual  dL/dK = {dK_manual}")


In [ ]:
# Verify against PyTorch autograd (this should pass once your manual code is correct)
x_t = torch.tensor([[[1.0, 2.0, 3.0, 4.0, 5.0]]])               # (B=1, C_in=1, L=5)
K_t = torch.tensor([[[0.5, -0.5, 1.0]]], requires_grad=True)    # (C_out=1, C_in=1, L=3)

y_t = F.conv1d(x_t, K_t)
loss = y_t.sum()       # this gives upstream gradient = ones at every position
loss.backward()

dK_torch = K_t.grad.squeeze().numpy()
print(f"PyTorch dL/dK = {dK_torch}")
try:
    print(f"\nManual matches PyTorch: {np.allclose(dK_manual, dK_torch)}")
except NameError:
    print("\nDefine dK_manual in the previous cell first.")


**Why this matters:** The fact that the kernel gradient is itself a convolution is exactly why PyTorch's `Conv2d.backward()` is implemented as another convolution call. The forward and backward operations have the same structure — only the operands change.

### Problem 12 — Max-pool gradient (sparse) [5 pts]

The max-pool gradient flows **only through the argmax position** of each window. Every other position in a window receives zero gradient. This is in contrast to conv layers, where the gradient is dense.


In [ ]:
# (DO NOT MODIFY the input)
X = torch.tensor([[[
    [1.0, 3.0, 2.0, 0.0],
    [5.0, 4.0, 1.0, 6.0],
    [0.0, 2.0, 7.0, 3.0],
    [1.0, 1.0, 5.0, 8.0],
]]], requires_grad=True)  # shape (1, 1, 4, 4) -- leaf tensor so X.grad works

# TODO: apply 2x2 max pooling with stride 2.
# y = ...

# TODO: define an upstream gradient delta of shape (1, 1, 2, 2) = [[1, 2], [3, 4]].
# delta = ...

# TODO: backpropagate the upstream gradient and inspect X.grad.
# y.backward(gradient=delta)
# print(X.grad.squeeze().numpy())
# Expected: a sparse 4x4 with a single nonzero per 2x2 window (at the argmax position).


### Problem 13 — Avg-pool gradient (dense) [5 pts]

For $k \times k$ average pooling with stride $k$, every input position in a window gets $1/k^2$ of that window's upstream gradient. The gradient is dense.


In [ ]:
# (DO NOT MODIFY the input)
X = torch.tensor([[[
    [1.0, 3.0, 2.0, 0.0],
    [5.0, 4.0, 1.0, 6.0],
    [0.0, 2.0, 7.0, 3.0],
    [1.0, 1.0, 5.0, 8.0],
]]], requires_grad=True)  # shape (1, 1, 4, 4) -- leaf tensor so X.grad works

# TODO: apply 2x2 AVERAGE pooling with stride 2.
# y = ...

# TODO: use the same upstream gradient as Problem 12, then backpropagate.
# delta = ...
# y.backward(gradient=delta)
# print(X.grad.squeeze().numpy())
# Expected: every position in each 2x2 window receives 1/4 of its window's
# upstream gradient. Top-left window: 0.25 everywhere; bottom-right: 1.0 everywhere.


**Practical consequence.** Max pool can leave "dead" positions in the input that never receive gradient (they are never the argmax). Average pool always updates every position, but its gradient signal is diluted by the window size.

---
## Bonus — End-to-end MNIST CNN

The math we just covered all comes together when training a real CNN. Below we build a small CNN, train it on MNIST for a few epochs, and inspect the learned filters.

This section is **optional** — if you skip the cells below, the rest of the notebook is self-contained.


In [ ]:
# Load MNIST
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# transforms.ToTensor() already normalizes pixel values to [0, 1].
transform = transforms.ToTensor()
train_ds = datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

print(f"Training:  {len(train_ds):>6,} samples")
print(f"Test:      {len(test_ds):>6,} samples")
print(f"One image: {train_ds[0][0].shape}  (channels, height, width)")


In [ ]:
# TODO: build a small CNN with this architecture:
#   Conv2d(1 -> 16, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool 2x2
#   Conv2d(16 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool 2x2
#   Flatten
#   Linear(32*7*7 -> 64) -> ReLU -> Dropout(0.25)
#   Linear(64 -> 10)
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define the layers as attributes.
        raise NotImplementedError("implement SmallCNN.__init__")

    def forward(self, x):
        # TODO: forward pass.
        raise NotImplementedError("implement SmallCNN.forward")


cnn = SmallCNN()
print(cnn)
print(f"\nTotal parameters: {count_params(cnn):,}")


In [ ]:
# Train for 3 epochs (enough to hit ~99% accuracy on MNIST)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = cnn.to(device)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f"Training on {device}")
print("-" * 60)
n_epochs = 3
for epoch in range(1, n_epochs + 1):
    # Train
    cnn.train()
    train_loss = 0.0
    for X_, y_ in train_loader:
        X_, y_ = X_.to(device), y_.to(device)
        optimizer.zero_grad()
        loss = criterion(cnn(X_), y_)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Evaluate
    cnn.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_, y_ in test_loader:
            X_, y_ = X_.to(device), y_.to(device)
            preds = cnn(X_).argmax(dim=1)
            correct += (preds == y_).sum().item()
            total   += y_.size(0)

    print(f"Epoch {epoch}/{n_epochs}: "
          f"train loss {train_loss / len(train_loader):.4f}  "
          f"test accuracy {100 * correct / total:.2f}%")


In [ ]:
# Show 8 sample predictions
cnn.eval()
X_sample, y_sample = next(iter(test_loader))
X_sample, y_sample = X_sample[:8], y_sample[:8]
with torch.no_grad():
    preds = cnn(X_sample.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_sample[i].squeeze(), cmap="gray")
    correct = preds[i].item() == y_sample[i].item()
    ax.set_title(f"true: {y_sample[i].item()}, pred: {preds[i].item()}",
                 color="black" if correct else "red")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# Visualize the learned first-layer filters (16 filters, each 3x3, 1 channel)
filters = cnn.conv1.weight.data.cpu().numpy()  # (16, 1, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(filters[i, 0], cmap="RdBu_r")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Learned 3x3 filters in conv1 (red = positive weight, blue = negative)")
plt.tight_layout()
plt.show()

print("Some filters look like edge detectors --- exactly the patterns")
print("we hand-designed in Problem 4. The network rediscovered them from data.")


---
## Done

You have now seen, in code, every concept covered by the homework PDF:

- Convolution from first principles, verified against PyTorch
- The output-size formula across many cases
- Hand-designed and learned edge detectors
- Padding modes
- Receptive field calculation through real architectures
- Pooling (max and average), both forward and backward
- LeNet-5 reconstructed exactly
- Why $3 \times 3$ filters dominate
- Backprop through conv and pool layers, manually and with autograd
- A real CNN trained on MNIST that converges to ~99% accuracy in three epochs

Cross-check the numerical answers in this notebook against the PDF's solution boxes — they should match exactly.
